In [1]:
import numpy as np
from torch.utils.data import Dataset, DataLoader

import transformers
from transformers import BertModel, BertTokenizer, get_scheduler, set_seed, AutoTokenizer, AutoModel

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

with open('/content/drive/MyDrive/Dataset/Sentiment/Hotel_ABSA/Hotel_ABSA/Train.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_train = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_train = df_train.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Hotel_ABSA/Hotel_ABSA/Test.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_test = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_test = df_test.drop(['id'], axis=1)

with open('/content/drive/MyDrive/Dataset/Sentiment/Hotel_ABSA/Hotel_ABSA/Dev.txt', 'r') as file:
    # Đọc nội dung của tệp và lưu vào biến content
    content = file.read()
    lines = content.split('\n\n')
    data = [line.split('\n') for line in lines]
    df_val = pd.DataFrame(data, columns=['id', 'text', 'label'])
    df_val = df_val.drop(['id'], axis=1)

In [3]:
import re
import torch

# Define the mappings
aspect_to_index = {'facilities#cleanliness': 0, 'facilities#comfort': 1, 'facilities#design&features': 2, 'facilities#general': 3, 'facilities#miscellaneous': 4,
                   'facilities#prices': 5, 'facilities#quality': 6, 'food&drinks#miscellaneous': 7, 'food&drinks#prices': 8, 'food&drinks#quality': 9,
                   'food&drinks#style&options': 10, 'hotel#cleanliness': 11, 'hotel#comfort': 12, 'hotel#design&features': 13, 'hotel#general': 14,
                   'hotel#miscellaneous': 15, 'hotel#prices': 16, 'hotel#quality': 17, 'location#general': 18, 'rooms#cleanliness': 19, 'rooms#comfort': 20,
                   'rooms#design&features': 21, 'rooms#general': 22, 'rooms#miscellaneous': 23, 'rooms#prices': 24, 'rooms#quality': 25, 'room_amenities#cleanliness': 26,
                   'room_amenities#comfort': 27, 'room_amenities#design&features': 28, 'room_amenities#general': 29, 'room_amenities#miscellaneous': 30, 'room_amenities#prices': 31,
                   'room_amenities#quality': 32, 'service#general': 33}
# Adding 'None' to handle unspecified sentiments for any detected aspect
sentiment_to_index = {'positive': 1, 'negative': 0, 'neutral': 0, 'none': 0}

# Preprocess the labels
def preprocess_labels(label_str):
    labels = re.findall(r'\{(.*?)\}', label_str.lower())
    aspects = torch.zeros(len(aspect_to_index), dtype=torch.long)
    sentiments = torch.full((len(aspect_to_index),), sentiment_to_index['none'], dtype=torch.long)

    # Initialize with 'None'
    for label in labels:
        if ', ' in label:
            aspect, sentiment = label.split(', ')
            if aspect in aspect_to_index and sentiment in sentiment_to_index:  # Only process known aspects with sentiments
                idx = aspect_to_index[aspect]
                aspects[idx] = 1
                sentiments[idx] = sentiment_to_index[sentiment]

    return aspects, sentiments

df_train['aspects'], df_train['sentiments'] = zip(*df_train['label'].apply(preprocess_labels))
df_test['aspects'], df_test['sentiments'] = zip(*df_test['label'].apply(preprocess_labels))
df_val['aspects'], df_val['sentiments'] = zip(*df_val['label'].apply(preprocess_labels))

In [4]:
from bs4 import BeautifulSoup

for df in [df_train, df_test, df_val]:
  for sentence in range(0, len(df.text)):
    # xóa tag, link http
    processed_feature = BeautifulSoup(str(df.text[sentence]), "html.parser").get_text()
    #email-id
    processed_feature = re.sub('b[w-]+?@w+?.w{2,4}b', 'emailadd', processed_feature)
    #url
    processed_feature = re.sub('(http[s]?S+)|(w+.[A-Za-z]{2,4}S*)', 'urladd', processed_feature)

    # Xóa tất cả các ký tự đặc biệt
    processed_feature = re.sub(r'\W', ' ', processed_feature)

    # xóa từ có chứa chữ số
    # processed_feature = re.sub('W*dw*', '', processed_feature)

    # Chuyển đổi sang chữ thường
    processed_feature = processed_feature.lower()

    # Thay thế nhiều khoảng trắng bằng một khoảng trắng
    processed_feature = re.sub(r'\s+', ' ', processed_feature, flags=re.I)

    # processed_features.append(processed_feature)
    df.text[sentence] = processed_feature

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.

  df.text[sentence] = processed_feature
<ipython-input-4-11490973c74e>:25: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df.text[sentence] = processed_feature
<ipython-input-4-1149097

In [5]:
df_train['aspects'] = df_train['aspects'].apply(lambda x: x.numpy())
df_train['sentiments'] = df_train['sentiments'].apply(lambda x: x.numpy())

df_test['aspects'] = df_test['aspects'].apply(lambda x: x.numpy())
df_test['sentiments'] = df_test['sentiments'].apply(lambda x: x.numpy())

df_val['aspects'] = df_val['aspects'].apply(lambda x: x.numpy())
df_val['sentiments'] = df_val['sentiments'].apply(lambda x: x.numpy())

In [6]:
df_train

,text,label,aspects,sentiments
0,vừa qua tôi có dùng dịch vụ tại khách sạn ttc ...,"{HOTEL#GENERAL, positive}, {SERVICE#GENERAL, p...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, ..."
1,tuy nhiên buffet sáng ở đây không được ngon và...,"{FOOD&DRINKS#QUALITY, negative}, {FOOD&DRINKS#...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,nhìn chung dịch vụ khách sạn cũng tốt chỉ có đ...,"{FOOD&DRINKS#QUALITY, negative}, {SERVICE#GENE...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,đội ngũ nhân viên phục vụ chu đáo tận tình,"{SERVICE#GENERAL, positive}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,trang thiết bị nội thất hài hoà tiện nghi hiện...,"{ROOM_AMENITIES#DESIGN&FEATURES, positive}, {R...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
...,...,...,...,...
7175,dù thái độ phục vụ bình thường nhưng phòng ở t...,"{SERVICE#GENERAL, neutral}, {ROOMS#DESIGN&FEAT...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7176,dịch vụ ở khách sạn mường thanh grand đà nẵng ...,"{SERVICE#GENERAL, positive}, {ROOMS#CLEANLINES...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7177,nhân viên phục vụ niềm nở vị trí đi lại thuận ...,"{SERVICE#GENERAL, positive}, {LOCATION#GENERAL...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
7178,khách sạn mường thanh đà nẵng tốt thôi chẳng c...,"{HOTEL#COMFORT, positive}","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ..."


In [7]:
from torch import cuda
device = 'cuda' if cuda.is_available() else 'cpu'

In [8]:
MAX_LEN = 128
TRAIN_BATCH_SIZE = 32
VALID_BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 5e-5
tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

In [9]:
class CustomDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.text = dataframe.text
        self.targets = self.data.aspects
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])
        text = " ".join(text.split())

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=False,
            max_length=self.max_len,
            pad_to_max_length=True,
            return_token_type_ids=False
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        # token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            # 'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }

In [10]:
training_set = CustomDataset(df_train, tokenizer, 128)
testing_set = CustomDataset(df_test, tokenizer, 128)
validation_set = CustomDataset(df_val, tokenizer, 128)

In [11]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [12]:
class AspectClass(torch.nn.Module):
    def __init__(self):
        super(AspectClass, self).__init__()
        self.l1 = transformers.AutoModel.from_pretrained('vinai/phobert-base')
        self.l2 = torch.nn.Dropout(0.3)
        self.l3 = torch.nn.Linear(768, 34)

    def forward(self, ids, mask):
        _, output_1= self.l1(ids, attention_mask = mask, return_dict=False)
        output_2 = self.l2(output_1)
        output = self.l3(output_2)
        return output

modelAspect = AspectClass()
modelAspect.to(device)

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

AspectClass(
  (l1): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNor

In [13]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [14]:
optimizer = torch.optim.AdamW(params =  modelAspect.parameters(), lr=LEARNING_RATE)

In [15]:
def train(epoch):
    modelAspect.train()
    for _,data in enumerate(training_loader, 0):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        # token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = modelAspect(ids, mask,)

        optimizer.zero_grad()
        loss = loss_fn(outputs, targets)
        if _%5000==0:
            print(f'Epoch: {epoch}, Loss:  {loss.item()}')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [16]:
for epoch in range(EPOCHS):
    train(epoch)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2834: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Epoch: 0, Loss:  0.6950729489326477
Epoch: 1, Loss:  0.1551242470741272
Epoch: 2, Loss:  0.10848254710435867
Epoch: 3, Loss:  0.07841857522726059
Epoch: 4, Loss:  0.0746907889842987
Epoch: 5, Loss:  0.0651974007487297
Epoch: 6, Loss:  0.050730157643556595
Epoch: 7, Loss:  0.0478120818734169
Epoch: 8, Loss:  0.03335171565413475
Epoch: 9, Loss:  0.03274022042751312


In [17]:
def validation(epoch):
    modelAspect.eval()
    fin_targets=[]
    fin_outputs=[]
    with torch.no_grad():
        for _, data in enumerate(testing_loader, 0):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            # token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = modelAspect(ids, mask)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())
    return fin_outputs, fin_targets

In [18]:
from sklearn import metrics

# for epoch in range(EPOCHS):
outputs, targets = validation(epoch)
outputs = np.array(outputs) >= 0.5
accuracy = metrics.accuracy_score(targets, outputs)
f1_score_micro = metrics.f1_score(targets, outputs, average='micro')
precision_score = metrics.precision_score(targets, outputs, average='micro')
recall_score = metrics.recall_score(targets, outputs, average='micro')
print(f"Accuracy Score = {accuracy}")
print(f"F1 Score (Micro) = {f1_score_micro}")
print(f"Precision = {precision_score}")
print(f"Recall = {recall_score}")

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2834: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Accuracy Score = 0.5990147783251232
F1 Score (Micro) = 0.7785450457684279
Precision = 0.8169868554095046
Recall = 0.743558282208589


In [19]:
class CustomDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.data = dataframe
        self.text = dataframe.text
        self.targets = self.data.sentiments
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = str(self.text[index])
        text = " ".join(text.split())

        inputs = self.tokenizer.encode_plus(
            text,
            None,
            add_special_tokens=False,
            max_length=self.max_len,
            pad_to_max_length=True,
            return_token_type_ids=False
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        # token_type_ids = inputs["token_type_ids"]


        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            # 'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.float)
        }

In [20]:
training_set = CustomDataset(df_train, tokenizer, 128)
testing_set = CustomDataset(df_test, tokenizer, 128)
validation_set = CustomDataset(df_val, tokenizer, 128)

In [21]:
train_params = {'batch_size': TRAIN_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': VALID_BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

training_loader = DataLoader(training_set, **train_params)
testing_loader = DataLoader(testing_set, **test_params)

In [22]:
class SentimentClass(torch.nn.Module):
    def __init__(self):
        super(SentimentClass, self).__init__()
        self.l1 = transformers.AutoModel.from_pretrained('vinai/phobert-base')
        self.l2 = torch.nn.Dropout(0.3)
        self.l3 = torch.nn.Linear(768, 34)

    def forward(self, ids, mask):
        _, output_1= self.l1(ids, attention_mask = mask, return_dict=False)
        output_2 = self.l2(output_1)
        output = self.l3(output_2)
        return output

modelSentiment = SentimentClass()
modelSentiment.to(device)

SentimentClass(
  (l1): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [23]:
def loss_fn(outputs, targets):
    return torch.nn.BCEWithLogitsLoss()(outputs, targets)

In [24]:
optimizer = torch.optim.AdamW(params =  modelSentiment.parameters(), lr=LEARNING_RATE)

In [25]:
def train(epoch):
    modelSentiment.train()
    for _,data in enumerate(training_loader, 0):
        ids = data['ids'].to(device, dtype = torch.long)
        mask = data['mask'].to(device, dtype = torch.long)
        # token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
        targets = data['targets'].to(device, dtype = torch.float)

        outputs = modelSentiment(ids, mask)

        optimizer.zero_grad()
        loss = loss_fn(outputs, targets)
        if _%5000==0:
            print(f'Epoch: {epoch}, Loss:  {loss.item()}')

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [26]:
for epoch in range(EPOCHS):
    train(epoch)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:2834: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Epoch: 0, Loss:  0.6965680718421936
Epoch: 1, Loss:  0.1299959123134613
Epoch: 2, Loss:  0.08797101676464081
Epoch: 3, Loss:  0.059076953679323196
Epoch: 4, Loss:  0.04822666943073273
Epoch: 5, Loss:  0.06080380454659462
Epoch: 6, Loss:  0.04104558005928993
Epoch: 7, Loss:  0.026493335142731667
Epoch: 8, Loss:  0.055788569152355194
Epoch: 9, Loss:  0.03677882254123688


In [27]:
def validation(epoch):
    modelSentiment.eval()
    fin_targets=[]
    fin_outputs=[]
    with torch.no_grad():
        for _, data in enumerate(testing_loader, 0):
            ids = data['ids'].to(device, dtype = torch.long)
            mask = data['mask'].to(device, dtype = torch.long)
            # token_type_ids = data['token_type_ids'].to(device, dtype = torch.long)
            targets = data['targets'].to(device, dtype = torch.float)
            outputs = modelSentiment(ids, mask)
            fin_targets.extend(targets.cpu().detach().numpy().tolist())
            fin_outputs.extend(torch.sigmoid(outputs).cpu().detach().numpy().tolist())
    return fin_outputs, fin_targets

In [28]:
from sklearn import metrics

# for epoch in range(EPOCHS):
outputs, targets = validation(epoch)
outputs = np.array(outputs) >= 0.5
accuracy = metrics.accuracy_score(targets, outputs)
f1_score_micro = metrics.f1_score(targets, outputs, average='micro')
precision_score = metrics.precision_score(targets, outputs, average='micro')
recall_score = metrics.recall_score(targets, outputs, average='micro')
print(f"Accuracy Score = {accuracy}")
print(f"F1 Score (Micro) = {f1_score_micro}")
print(f"Precision = {precision_score}")
print(f"Recall = {recall_score}")

Accuracy Score = 0.6438423645320197
F1 Score (Micro) = 0.7623450304685858
Precision = 0.7897257292120157
Recall = 0.7367993501218522
